# TAE-IA · Module 6 · L03 — Image-to-Image: Transform with Text Guidance

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Lesson** | L03 |
| **Track** | A — Vision |
| **Estimated duration** | 2 hours |
| **GPU required** | T4 (Colab) |
| **Prerequisites** | L01 and L02 completed |

## Learning objectives
By the end of this notebook you will be able to:
- [ ] Explain how img2img differs from txt2img at the latent level
- [ ] Use `StableDiffusionImg2ImgPipeline` to transform an image with a text prompt
- [ ] Predict and verify the effect of `strength` on the output
- [ ] Create a sketch-to-render pipeline using Canny edge detection

## Before you start
- L01 and L02 completed
- T4 GPU runtime selected (`Runtime > Change runtime type > T4 GPU`)

---

## Cell 0 — Setup (always run this first)

> Mounts Drive, checks GPU, fixes seed, and logs in to HuggingFace.

In [ ]:
# ================================================================
# TAE-IA M6 · Standard Setup -- DO NOT MODIFY THIS CELL
# ================================================================
import os, sys, random, shutil
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

MODEL_CACHE = '/content/drive/MyDrive/TAE_IA_M6/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
print(f'Model cache: {MODEL_CACHE}')

# Clear any cache left by earlier course versions.
for _leftover in ('hub', 'xet'):
    _p = os.path.join(MODEL_CACHE, _leftover)
    if os.path.exists(_p):
        shutil.rmtree(_p)

import torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Go to: Runtime > Change runtime type > T4 GPU')
    raise SystemExit('GPU required.')

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f'Fixed seed: {SEED}')
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')

# HuggingFace login -- needed every new Colab session
import huggingface_hub

try:
    _token = huggingface_hub.get_token()
except Exception:
    _token = None

if _token:
    print(f"Already logged in to HuggingFace as: {huggingface_hub.whoami()['name']}")
else:
    try:
        from google.colab import userdata
        _hf_token = userdata.get('HF_TOKEN')
    except Exception:
        _hf_token = None

    if _hf_token:
        huggingface_hub.login(token=_hf_token, add_to_git_credential=False)
        print(f"Logged in to HuggingFace as: {huggingface_hub.whoami()['name']}")
    else:
        raise RuntimeError(
            'No HF_TOKEN found in Colab Secrets (key icon, left sidebar).\n'
            'Add a secret named HF_TOKEN with your HuggingFace read token, enable notebook access, '
            'then re-run this cell.\n'
            'See L00 Cell 4 if you need to generate a token or accept the SD 1.5 license.'
        )

In [ ]:
# ================================================================
# Install dependencies for L03
# ================================================================
!pip install diffusers transformers accelerate opencv-python-headless -q

import diffusers, cv2
print(f'diffusers {diffusers.__version__}  |  OpenCV {cv2.__version__}')

---
## Part 1 — Context and Key Concepts

> Read this before running any code.

### How img2img differs from txt2img

In txt2img, the UNet starts from **pure Gaussian noise** in latent space and denoises it step by step, guided only by the text prompt. The output has no relation to any starting image.

In img2img:
1. The source image is **encoded into latent space** with the VAE encoder
2. A controlled amount of noise is **added** to that latent (controlled by `strength`)
3. The UNet runs only the remaining denoising steps — it never reaches pure noise and recovers from scratch

```
txt2img:   [pure noise]  →  UNet × 25 steps  →  VAE decode  →  image

img2img:   [source image]  →  VAE encode  →  add noise (strength)  →  UNet × (strength×25)  →  VAE decode  →  image
```

### The `strength` parameter

`strength` controls how much noise is injected before denoising begins:

| `strength` | Steps run (of 25) | Effect |
|---|---|---|
| 0.0 | 0 | Source returned unchanged |
| 0.3 | ~7 | Subtle style shift, strong structure retention |
| 0.6 | ~15 | Significant change, composition mostly intact |
| 0.9 | ~22 | Near-complete transformation, faint trace of source |
| 1.0 | 25 | Equivalent to txt2img — source ignored |

A good **starting point** is `strength=0.7`. Lower values preserve more of the original; higher values give the model more creative freedom.

### The `.components` trick

`StableDiffusionPipeline` and `StableDiffusionImg2ImgPipeline` share the **exact same weights** (UNet, VAE, CLIP, scheduler). Once you have one loaded, you can create the other without reloading:

```python
pipe_img = StableDiffusionImg2ImgPipeline(**pipe_txt.components)
```

This avoids an extra ~4 GB load from Drive and keeps VRAM usage identical.

### Sketch-to-render

Canny edge detection extracts the structural contours of an image as black-on-white lines. When used as the `image` input for img2img with high `strength` (0.8–0.95), the model preserves the spatial layout implied by the edges while filling in all detail from the prompt. This is the classic **sketch-to-render** or **wireframe-to-photo** workflow.

---

## Part 2 — Lab

### Section 2.1 — Load pipeline and generate source image

We generate a source image with txt2img first, then reuse the same weights for img2img.

In [ ]:
# Section 2.1 — Load txt2img, generate source, switch to img2img
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
from huggingface_hub import snapshot_download
import torch, os
import matplotlib.pyplot as plt

OUTPUT_DIR = '/content/drive/MyDrive/TAE_IA_M6/L03_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def gen(seed=SEED):
    return torch.Generator("cuda").manual_seed(seed)

# Fetch only the fp16 weights + configs (~2.5 GB), not every format in the repo.
SD15_DIR = os.path.join(MODEL_CACHE, 'sd15-local')
snapshot_download(
    "runwayml/stable-diffusion-v1-5",
    local_dir=SD15_DIR,
    allow_patterns=["*.json", "*.txt", "*.fp16.safetensors"],
)

# Load txt2img
pipe_txt = StableDiffusionPipeline.from_pretrained(
    SD15_DIR,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")
print("txt2img pipeline loaded.")

# Generate a reproducible source image
SOURCE_PROMPT = "a mountain village in autumn, warm golden light, photorealistic, high detail"

source = pipe_txt(
    SOURCE_PROMPT,
    num_inference_steps = 25,
    guidance_scale      = 7.5,
    generator           = gen()
).images[0]

source.save(os.path.join(OUTPUT_DIR, "l03_source.png"))
print(f"Source image saved: l03_source.png  ({source.size[0]}×{source.size[1]} px)")

# Reuse the same weights for img2img — no extra download or VRAM
pipe_img = StableDiffusionImg2ImgPipeline(**pipe_txt.components)
print("img2img pipeline ready (shared weights).")

source

**What do you observe?**  
- How long did loading take compared to L01 (cached vs. fresh download)?
- Describe the source image in one sentence — this is your baseline for all experiments below.

*Write your observation here:*

(double-click to edit)

### Section 2.2 — Strength comparison (0.3 / 0.6 / 0.9)

Same prompt and seed throughout. Only `strength` changes.

In [ ]:
# Section 2.2 — Effect of strength
STYLE_PROMPT = "impressionist oil painting, vibrant autumn colors, visible brushstrokes, Monet style"
strengths    = [0.3, 0.6, 0.9]

strength_results = []
for s in strengths:
    img = pipe_img(
        prompt         = STYLE_PROMPT,
        image          = source,
        strength       = s,
        guidance_scale = 7.5,
        generator      = gen()
    ).images[0]
    fname = os.path.join(OUTPUT_DIR, f"strength_{str(s).replace('.','')}.png")
    img.save(fname)
    strength_results.append((s, img))
    print(f"strength={s}  →  {fname}")

# Display source + all three results
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(source); axes[0].set_title("Source (txt2img)", fontsize=10); axes[0].axis('off')
for ax, (s, img) in zip(axes[1:], strength_results):
    ax.imshow(img); ax.set_title(f"strength={s}", fontsize=10); ax.axis('off')
plt.suptitle("img2img strength comparison — same prompt and seed", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "grid_strength.png"), dpi=100)
plt.show()

**What do you observe?**  
- At `strength=0.3`: which elements of the source (shapes, colors, composition) are still recognizable?
- At `strength=0.9`: is there any trace of the original mountain village?
- At what `strength` value does the transition feel most dramatic?

*Write your observation here:*

(double-click to edit)

### Section 2.3 — Sketch to realistic

Convert the source image to a Canny edge sketch, then use img2img to render it into a detailed scene.

In [ ]:
# Section 2.3 — Sketch-to-realistic via Canny edge detection
import cv2
import numpy as np
from PIL import Image

# Step 1: Extract edges from the source
img_gray = np.array(source.convert("L"))              # grayscale
edges    = cv2.Canny(img_gray, threshold1=50, threshold2=150)
sketch   = Image.fromarray(255 - edges)               # invert: black edges on white background
sketch   = sketch.convert("RGB")                      # pipeline expects RGB
sketch.save(os.path.join(OUTPUT_DIR, "sketch.png"))
print("Sketch saved.")

# Step 2: Render the sketch with img2img
RENDER_PROMPT = (
    "photorealistic mountain village in autumn, detailed stone walls, "
    "warm golden hour light, crisp mountain air, high detail"
)

rendered = pipe_img(
    prompt         = RENDER_PROMPT,
    image          = sketch,
    strength       = 0.85,
    guidance_scale = 8.0,
    generator      = gen()
).images[0]
rendered.save(os.path.join(OUTPUT_DIR, "sketch_rendered.png"))
print("Rendered image saved.")

# Display source → sketch → render
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (title, img) in zip(axes, [("Source", source), ("Canny Sketch", sketch), ("Rendered", rendered)]):
    ax.imshow(img); ax.set_title(title, fontsize=11); ax.axis('off')
plt.suptitle("Sketch-to-realistic pipeline", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sketch_pipeline.png"), dpi=100)
plt.show()

**What do you observe?**  
- Which structural elements from the sketch (building outlines, horizon, shapes) appear in the final render?
- Where did the model deviate from the edge map? Why do you think it diverged there?
- What would happen if you lowered `strength` to 0.6 on the sketch?

*Write your observation here:*

(double-click to edit)

### Section 2.4 — img2img vs txt2img: direct comparison

Same prompt, same seed. txt2img starts from random noise; img2img starts from the encoded source.

In [ ]:
# Section 2.4 — Direct comparison: txt2img vs img2img
COMPARE_PROMPT = "watercolor landscape, soft blues and greens, misty mountains, gentle brushwork"

# txt2img — starts from pure Gaussian noise
img_t2i = pipe_txt(
    COMPARE_PROMPT,
    num_inference_steps = 25,
    guidance_scale      = 7.5,
    generator           = gen()
).images[0]
img_t2i.save(os.path.join(OUTPUT_DIR, "compare_txt2img.png"))
print("txt2img done.")

# img2img — starts from the encoded source
img_i2i = pipe_img(
    prompt         = COMPARE_PROMPT,
    image          = source,
    strength       = 0.7,
    guidance_scale = 7.5,
    generator      = gen()
).images[0]
img_i2i.save(os.path.join(OUTPUT_DIR, "compare_img2img.png"))
print("img2img done.")

# Display source / txt2img / img2img
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (title, img) in zip(axes, [
    ("Source (photorealistic)", source),
    ("txt2img (from noise)", img_t2i),
    ("img2img (from source, s=0.7)", img_i2i)
]):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle(f'Prompt: "{COMPARE_PROMPT[:60]}..."', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "compare_t2i_vs_i2i.png"), dpi=100)
plt.show()

**What do you observe?**  
- Does the img2img output preserve the composition of the source (village position, mountains, sky)?
- How does txt2img's scene compare — is it a similar composition or something different?
- For a client project where you need to stylize an existing photo, which would you use and why?

*Write your observation here:*

(double-click to edit)

### Section 2.5 — Batch img2img: one source, many styles

Same source, same seed, same `strength` — only the style prompt changes across the loop.

In [ ]:
# Section 2.5 — Batch img2img: one source, many styles
styles = [
    "watercolor landscape, soft pastels",
    "oil painting, dramatic chiaroscuro lighting",
    "cyberpunk neon, rain-slicked streets",
    "ukiyo-e woodblock print, flat colour fields",
]

batch_results = []
for style in styles:
    img = pipe_img(
        prompt         = f"a mountain village, {style}",
        image          = source,
        strength       = 0.7,
        guidance_scale = 7.5,
        generator      = gen()
    ).images[0]
    fname = os.path.join(OUTPUT_DIR, f"village_{style[:10].replace(' ', '_')}.png")
    img.save(fname)
    batch_results.append((style, img))
    print(f"Style done: {style}")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (style, img) in zip(axes, batch_results):
    ax.imshow(img); ax.set_title(style, fontsize=9, wrap=True); ax.axis('off')
plt.suptitle("Batch img2img — same source, seed, and strength; only the style prompt changes", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "grid_batch_styles.png"), dpi=100)
plt.show()

**What do you observe?**  
- Which style diverged most from "a mountain village" as a recognizable subject?
- Fixing the source, seed, and strength isolates the prompt as the only variable — does the grid actually look like a controlled comparison, or do you see other differences creeping in?

*Write your observation here:*

(double-click to edit)

---
## Part 3 — Exercises

### Exercise 1 — Find your strength sweet spot

**Task:** Choose a different style prompt (not impressionist painting — pick something clearly different: anime, pencil sketch, Renaissance portrait, cyberpunk, etc.). Run img2img on the source at `strength = 0.4, 0.6, 0.75, 0.9`. Show all four results in a grid with the source image.

**Expected output:** A 5-image grid (source + 4 strength values). In a markdown cell, state which `strength` value you would use in production for this style and why.

In [ ]:
# Exercise 1 -- your code here
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise1.png"))
MY_STYLE_PROMPT = "..."   # replace with your chosen style
MY_STRENGTHS    = [0.4, 0.6, 0.75, 0.9]

# ...

*Which strength value would you use for this style and why?*

(double-click to edit)

### Exercise 2 — Iterative refinement

**Task:** Take the best `img2img` result from Exercise 1 and use it as the **source** for a second img2img pass with a different prompt and `strength=0.4`. Show the full chain: original source → first pass → second pass.

**Expected output:** A 3-image chain. In a markdown cell, describe what changed at each step and whether the second pass improved or degraded the result.

In [ ]:
# Exercise 2 -- your code here
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise2.png"))
# Pass 1 result = your best image from Exercise 1
# Pass 2: use pass_1_result as source, strength=0.4, different prompt
# ...

*What changed at each step? Did the second pass improve the result?*

(double-click to edit)

---
## Part 4 — Critical Analysis

> Required. Answer with real outputs from today's session — not hypothetical answers.

**4.1 — Strength threshold: at what `strength` value did the output start to visibly differ from the source? Show the two images on either side of that threshold (e.g., 0.3 and 0.6).**

*Write here (reference your Section 2.2 grid):*


---

**4.2 — Sketch fidelity: which structural elements from the Canny sketch appeared in the rendered output? Name at least two elements that were preserved and one that was not.**

*Write here (be specific — e.g., "the roofline at the top-left was preserved, but the window positions were ignored"):*


---

**4.3 — Composition comparison: in the img2img vs txt2img comparison, which elements of the source composition were preserved and which were discarded?**

*Write here (compare horizon line, subject position, color balance, etc.):*


---

**4.4 — Real-world choice: describe a concrete task (not a class demo) where img2img would produce a better result than txt2img. What would the source image be, and at what `strength` would you run it?**

*Write here (be specific — e.g., "a product photo being converted to a hand-drawn illustration for a brand identity package, source = product photo, strength ≈ 0.65"):*


---
## Submission Checklist

- [ ] All cells ran from start to finish without errors
- [ ] Source image saved (`l03_source.png`) — needed as inpainting input in L04
- [ ] Section 2.2 strength grid saved (`grid_strength.png`)
- [ ] Section 2.3 sketch pipeline saved (`sketch_pipeline.png`)
- [ ] Section 2.4 txt2img vs img2img comparison saved (`compare_t2i_vs_i2i.png`)
- [ ] Section 2.5 batch style grid saved (`grid_batch_styles.png`)
- [ ] Exercise 1 — strength sweep with your chosen style
- [ ] Exercise 2 — iterative refinement chain (3 images)
- [ ] Part 4 — Critical Analysis completed (all 4 questions with real evidence)
- [ ] All outputs saved to `TAE_IA_M6/L03_output/` on Drive

**Save:** `File > Save a copy in Drive`

---
## Before You Close This Tab

- [ ] Confirmed all outputs from this session are saved in `TAE_IA_M6/L03_output/` on Drive (see checklist above)
- [ ] Disconnected and deleted this runtime: `Runtime > Disconnect and delete runtime`

Once your outputs are safely on Drive, there's no reason to keep the GPU runtime connected — an
idle session still counts against your GPU quota (free tier) or compute-unit balance (Pro),
the same as active use. Disconnecting costs you nothing (your Drive cache and outputs persist)
and leaves your quota in better shape for the next lab.

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L03*  
*Platform: Google Colab (T4 GPU) · Python 3.10*